# GRU Forecasting of Electricity Load

Train a small GRU network on the electricity-temperature dataset to forecast hourly electricity load one step ahead, using both load and temperature as inputs. Compare predictions against the held-out test window and inspect the ACF/PACF of the residuals.

In [ ]:
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_percentage_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping

from PythonTsa.datadir import getdtapath
from PythonTsa.TsTensor import tstensor, create_evaluation_df
from PythonTsa.plot_acf_pacf import acf_pacf_fig

In [ ]:
# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# 1. Load data
dtapath = getdtapath()
tsdta = pd.read_csv(dtapath + 'elec-temp.csv')
tsdta['time'] = pd.to_datetime(tsdta['time'])
tsdta.set_index('time', inplace=True)

In [ ]:
# 2. Train / validation / test split
valid_stdta = '2014-09-01 00:00:00'
test_stdta  = '2014-11-01 00:00:00'
T       = 6   # input-window length (hours)
HORIZON = 1   # forecast horizon (hours)

train      = tsdta.loc[:valid_stdta].copy()
valid_part = tsdta.loc[valid_stdta:test_stdta].copy()
test_part  = tsdta.loc[test_stdta:].copy()

In [ ]:
# 3. Scaling
y_scaler = MinMaxScaler()
y_scaler.fit(train[['load']])

X_scaler = MinMaxScaler()
train[['load', 'temp']] = X_scaler.fit_transform(train[['load', 'temp']])

In [ ]:
# 4. Tensor preparation
tensor = {'X': (range(-T + 1, 1), ['load', 'temp'])}

ts_train_inp = tstensor(
    dataset=train,
    target='load',
    h=HORIZON,
    tensor_structure=tensor,
    freq='h',
    drop_incomplete=True,
)

In [ ]:
# 5. Validation-set tensors
back_ts_data = dt.datetime.strptime(valid_stdta, '%Y-%m-%d %H:%M:%S') - dt.timedelta(hours=T - 1)
tsdta_valid  = tsdta.loc[back_ts_data:test_stdta, ['load', 'temp']].copy()
tsdta_valid[['load', 'temp']] = X_scaler.transform(tsdta_valid)

valid_inputs = tstensor(tsdta_valid, 'load', HORIZON, tensor)

In [ ]:
# 6. GRU model
LATENT_DIM = 5
BATCH_SIZE = 32
EPOCHS     = 40

model = Sequential([
    GRU(LATENT_DIM, input_shape=(T, 2)),
    Dense(HORIZON),
])
model.compile(optimizer='adam', loss='mse')

earlystop = EarlyStopping(monitor='val_loss', patience=5, min_delta=0)

model_history = model.fit(
    ts_train_inp['X'],
    ts_train_inp['target'],
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(valid_inputs['X'], valid_inputs['target']),
    callbacks=[earlystop],
    verbose=1,
)

In [ ]:
# 7. Test-set tensors & prediction
back_tsdta = dt.datetime.strptime(test_stdta, '%Y-%m-%d %H:%M:%S') - dt.timedelta(hours=T - 1)
tsdta_test = tsdta.loc[back_tsdta:, ['load', 'temp']].copy()
tsdta_test[['load', 'temp']] = X_scaler.transform(tsdta_test)

tsdta_test_inputs = tstensor(tsdta_test, 'load', HORIZON, tensor)
ts_predictions    = model.predict(tsdta_test_inputs['X'])

In [ ]:
# 8. Evaluation
ev_tsdta = create_evaluation_df(ts_predictions, tsdta_test_inputs, HORIZON, y_scaler)

print(ev_tsdta.head())
print('MAPE:', mean_absolute_percentage_error(ev_tsdta['actual'], ev_tsdta['prediction']))

In [ ]:
# 9. Plot prediction vs actual for an early slice of the test window
ev_tsdta[ev_tsdta.timestamp < '2014-11-08'].plot(
    x='timestamp',
    y=['prediction', 'actual'],
    style=['--r', '-b'],
)
plt.ylabel('Electricity load')
plt.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.savefig('gru_model.png', transparent=True, bbox_inches='tight')
plt.show()

In [ ]:
# 10. Residual diagnostics
resid = ev_tsdta['actual'] - ev_tsdta['prediction']
acf_pacf_fig(resid, lag=72)
plt.savefig('acf_pacf.png', transparent=True, bbox_inches='tight')
plt.show()